TODO: try evaluation with qwen2 and qwen2.5

# EPIC-KITCHENS 55 MCQ Inference with Qwen3-VL

This notebook performs Multiple Choice Question (MCQ) inference on the EPIC-KITCHENS dataset using the Qwen3-VL model via Unsloth.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp -r /content/drive/MyDrive/DL25_Project/EPIC-KITCHENS55-SAMPLED-FRAMES/EPIC-KITCHENS55-SAMPLED-FRAMES.zip /content

In [ ]:
!unzip -q /content/EPIC-KITCHENS55-SAMPLED-FRAMES.zip -d /content/

In [ ]:
#!cp -r /content/drive/MyDrive/qwen3_2b_finetune_with_vision /content

In [ ]:
#!cp -r /content/drive/MyDrive/qwen2_2b_with_vision /content
!cp -r /content/drive/MyDrive/qwen2_2b /content

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.57.1
!pip install --no-deps trl==0.22.2

In [ ]:
import pandas as pd

In [ ]:
from unsloth import FastVisionModel
import json
import random
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
# Configuration
JSONL_PATH = "/content/avion_distractors_combined.jsonl"
FRAMES_ROOT = "/content/content/EPIC-KITCHENS55-SAMPLED-FRAMES"
SPLIT_CSV_PATH = "/content/ek55_data_split.csv"
OUTPUT_PATH = "ek55_mcq_inference_results.json"
#MODEL_ID = "unsloth/Qwen2.5-VL-3B-Instruct-unsloth-bnb-4bit" - 59.27%
#MODEL_ID = "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit" - 66.57%
#MODEL_ID = "unsloth/Qwen3-VL-4B-Instruct-unsloth-bnb-4bit" - 67.48%
#MODEL_ID = "unsloth/Qwen3-VL-2B-Instruct-unsloth-bnb-4bit" - 55.62%
# MODEL_ID = "unsloth/Qwen2-VL-2B-Instruct-unsloth-bnb-4bit" - 57.45%
MODEL_ID = "qwen2_2b_with_vision"
NUM_FRAMES = 16
# MAX_SAMPLES = 329
TARGET_SPLIT = "test"

In [ ]:
# Load Model
model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_ID,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

==((====))==  Unsloth 2025.12.9: Fast Qwen3_Vl patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.41G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/213 [00:00<?, ?B/s]

In [ ]:
def load_jsonl(path):
    with open(path, 'r') as f:
        return [json.loads(line) for line in f]

def get_frame_path(root, participant_id, video_id, frame_idx):
    # Path structure: root/P01/rgb/P01_01/frame_0000000001.jpg
    # Note: Using 10-digit zero padding for frame index as verified
    filename = f"frame_{frame_idx:010d}.jpg"
    return os.path.join(root, participant_id, "rgb", video_id, filename)

def format_options(ground_truth_narration, distractors):
    options = [ground_truth_narration] + distractors
    random.shuffle(options)

    option_labels = ['A', 'B', 'C', 'D', 'E']
    # Ensure we don't have more options than labels
    options = options[:len(option_labels)]

    formatted_options_str = ""
    correct_option_label = None

    for label, text in zip(option_labels, options):
        formatted_options_str += f"{label}. {text}\n"
        if text == ground_truth_narration:
            correct_option_label = label

    return formatted_options_str, correct_option_label, options

In [ ]:
# Load Data & Filter by Split
split_df = pd.read_csv(SPLIT_CSV_PATH)
video_to_split = dict(zip(split_df['video_id'], split_df['split']))
full_data = load_jsonl(JSONL_PATH)

data = []
skipped = 0
for entry in full_data:
    vid = entry['video_id']

    split = video_to_split.get(vid, 'test')

    if split == TARGET_SPLIT:
        data.append(entry)
    else:
        skipped += 1

print(f"Filtered dataset for split '{TARGET_SPLIT}'")
print(f"Kept: {len(data)} samples")
print(f"Skipped: {skipped} samples")


Filtered dataset for split 'test'
Kept: 1240 samples
Skipped: 5102 samples


In [ ]:
# Main Inference Loop
results = []

system_prompt = "You are an expert in video and action recognition. Answer with ONLY the option letter (A, B, C, D, or E). Do not explain."

for item in tqdm(data):
    uid = item['uid']
    participant_id = item['participant_id']
    video_id = item['video_id']
    frame_indices = item['frame_indices']
    ground_truth = item['ground_truth']['narration']

    # Extract answer text from the list of dicts and take top 4
    distractors = [d['answer'] for d in item['distractors_with_confidence'][:4]]

    # Use provided frame indices (pre-sampled)
    sampled_indices = frame_indices


    # Load images
    images = []
    valid_sample = True
    for idx in sampled_indices:
        path = get_frame_path(FRAMES_ROOT, participant_id, video_id, idx)
        if os.path.exists(path):
            try:
                img = Image.open(path).convert("RGB")
                images.append(img)
            except Exception as e:
                print(f"Error loading image {path}: {e}")
                valid_sample = False
                break
        else:
            print(f"Image not found: {path}")
            valid_sample = False
            break

    if not valid_sample or not images:
        print(f"Skipping sample {uid} due to missing/bad frames")
        continue

    # Format Prompt
    options_str, correct_label, shuffled_options = format_options(ground_truth, distractors)
    user_prompt = f"The frames are given in a temporal order. Select the most suitable action from the options.\n\nOptions:\n{options_str}\n"

    # Construct Messages
    messages = [
        {
            "role": "user",
            "content": [
                # Add images
                *[{"type": "image", "image": img} for img in images],
                # Add text
                {"type": "text", "text": user_prompt}
            ]
        }
    ]

    if system_prompt:
        messages.insert(0, {"role": "system", "content": system_prompt})

    # Prepare inputs
    # input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(
        images,
        input_text,
        add_special_tokens=False,
        return_tensors="pt",
    ).to("cuda")

    # Inference
    FastVisionModel.for_inference(model)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=10, # We only expect a single letter answer
            use_cache=True,
            temperature=0.1,
            min_p = 0.1
        )

    # Decode
    # The generated tokens are appended to input_ids. We need to slice.
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, output_ids)]
    output_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    # Clean output (take first letter)
    model_choice = output_text.strip().split()[0].replace('.', '').replace(')', '') if output_text else ""

    # Store results
    result_entry = {
        "uid": uid,
        "participant_id": participant_id,
        "video_id": video_id,
        "frame_indices": frame_indices,
        "sampled_indices": sampled_indices,
        "ground_truth": ground_truth,
        "ground_truth_option": correct_label,
        "distractors": distractors,
        "options_presented": shuffled_options,
        "model_choice": model_choice,
        "model_option": model_choice, # redundant but requested fields
        "raw_output": output_text
    }
    results.append(result_entry)

    print(f"UID: {uid} | GT: {correct_label} | Pred: {model_choice} | Correct: {model_choice == correct_label}")

  0%|          | 1/1240 [00:01<32:38,  1.58s/it]

UID: 2809 | GT: A | Pred: A | Correct: True


  0%|          | 2/1240 [00:02<30:03,  1.46s/it]

UID: 2810 | GT: A | Pred: A | Correct: True


  0%|          | 3/1240 [00:04<29:34,  1.43s/it]

...

UID: 19735 | GT: B | Pred: B | Correct: True


100%|█████████▉| 1239/1240 [29:39<00:01,  1.45s/it]

UID: 19736 | GT: A | Pred: A | Correct: True


100%|██████████| 1240/1240 [29:41<00:00,  1.44s/it]

UID: 19737 | GT: D | Pred: D | Correct: True


In [ ]:
# Save Results
with open(OUTPUT_PATH, 'w') as f:
    json.dump(results, f, indent=4)

print(f"Results saved to {OUTPUT_PATH}")

Results saved to ek55_mcq_inference_results.json


In [ ]:
# Calculate and Print Overall Accuracy

correct_count = 0
total_count = len(results)

for res in results:
    if res.get('model_choice') == res.get('ground_truth_option'):
        correct_count += 1

accuracy = (correct_count / total_count) * 100 if total_count > 0 else 0
print("=" * 30)
print(f"Total Samples: {total_count}")
print(f"Correct Predictions: {correct_count}")
print(f"Overall Accuracy: {accuracy:.2f}%")
print("=" * 30)


Total Samples: 1240
Correct Predictions: 1065
Overall Accuracy: 85.89%
